# Sentinel-1元数据季度批量下载工具 (2014-2024)

本Notebook提供以下功能：
1. 按季度批量下载2014-2024年Sentinel-1元数据
2. 每年的四个季度数据保存在对应年份的文件夹下
3. 每年的所有季度数据合并为一个年度文件
4. 包含错误处理和重试机制

In [1]:
import asf_search as asf
import pandas as pd
import os
import datetime
from dateutil.parser import parse
from datetime import datetime, timedelta
import numpy as np
import traceback
import time
import sys

# 定义研究区域的多边形 (Arctic Canada North)
STUDY_AREA = "POLYGON ((-125.0 74.0, -75.0 74.0, -75.0 77.0, -74.73 77.51, -74.28 78.06, -73.80 78.49, -73.11 78.81, -72.25 79.11, -71.01 79.39, -70.08 79.64, -69.35 79.94, -68.64 80.3, -67.8 80.56, -67.02 80.78, -66.06 80.95, -65.13 81.13, -64.33 81.27, -63.37 81.5, -62.55 81.72, -61.86 81.93, -60.86 82.1, -60.3 82.3, -60.0 82.5, -60.0 85.0, -125.0 85.0, -125.0 74.0))"

# 创建基础输出目录
BASE_OUTPUT_DIR = r"E:\NWP\Sentinel1\Metadata"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

# 定义季度
QUARTERS = {
    1: (1, 3),  # Q1: 1月-3月
    2: (4, 6),  # Q2: 4月-6月
    3: (7, 9),  # Q3: 7月-9月
    4: (10, 12)  # Q4: 10月-12月
}


## 1. 按季度下载Sentinel-1元数据

In [2]:
# 定义按季度下载的函数
def download_sentinel1_metadata_by_quarter(year, quarter, polygon=STUDY_AREA, max_results=10000, max_retries=3):
    # 获取季度的开始和结束月份
    start_month, end_month = QUARTERS[quarter]
    
    # 计算季度的开始和结束日期
    start_date = f"{year}-{start_month:02d}-01T00:00:00Z"
    
    # 计算结束日期（下一个季度的第一天）
    if end_month == 12:
        next_year = year + 1
        next_month = 1
    else:
        next_year = year
        next_month = end_month + 1
        
    end_date = f"{next_year}-{next_month:02d}-01T00:00:00Z"
    
    # 创建年份目录
    year_dir = os.path.join(BASE_OUTPUT_DIR, f"{year}")
    os.makedirs(year_dir, exist_ok=True)
    
    # 设置搜索参数
    opts = asf.ASFSearchOptions(**{
        "maxResults": max_results,
        "intersectsWith": polygon,
        "start": start_date,
        "end": end_date,
        "dataset": ["SENTINEL-1"]
    })
    
    # 执行查询，带有重试机制
    for retry in range(max_retries):
        try:
            print(f"正在查询 {year}年第{quarter}季度 (Q{quarter}: {start_month}-{end_month}月) 的Sentinel-1数据...")
            results = asf.search(opts=opts)
            print(f"找到 {len(results)} 条记录")
            break
        except Exception as e:
            if retry < max_retries - 1:
                wait_time = (retry + 1) * 5  # 递增等待时间
                print(f"查询出错: {e}\n等待 {wait_time} 秒后重试...")
                time.sleep(wait_time)
            else:
                print(f"查询失败，已达到最大重试次数: {e}")
                print(traceback.format_exc())
                return None
    
    if len(results) == 0:
        print(f"⚠️ {year}年第{quarter}季度没有找到数据")
        return None
    
    # 提取属性并转换为DataFrame
    data = []
    for r in results:
        try:
            # 提取几何信息
            geom = r.geometry['coordinates'][0]
            near_end = geom[0]
            near_start = geom[1]
            far_start = geom[2]
            far_end = geom[3]
            
            # 提取属性
            record = {
                "Granule Name": r.properties.get('sceneName', ''),
                'Platform': r.properties.get('platform', ''),
                'Sensor': r.properties.get('sensor', 'C-SAR'),
                'Beam Mode': r.properties.get('beamModeType', ''),
                'Start Time': r.properties.get('startTime', ''),
                'End Time': r.properties.get('stopTime', ''),
                'Orbit': r.properties.get('orbit', ''),
                'Path Number': r.properties.get('pathNumber', ''),
                'Processing Date': r.properties.get('processingDate', ''),
                'Processing Level': r.properties.get('processingLevel', ''),
                'Center Lat': r.properties.get('centerLat', ''),
                'Center Lon': r.properties.get('centerLon', ''),
                'Near Start Lat': near_start[1], 'Near Start Lon': near_start[0],
                'Far Start Lat': far_start[1], 'Far Start Lon': far_start[0],
                'Far End Lat': far_end[1], 'Far End Lon': far_end[0],
                'Near End Lat': near_end[1], 'Near End Lon': near_end[0],
                'Ascending or Descending': r.properties.get('flightDirection', ''),
                'URL': r.properties.get('url', '')
            }
            
            data.append(record)
        except Exception as e:
            print(f"⚠️ 处理记录时出错: {e}")
            continue
    
    if not data:
        print(f"⚠️ {year}年第{quarter}季度没有有效数据可处理")
        return None
        
    df = pd.DataFrame(data)
    
    # 保存为CSV文件
    filename = f"sentinel1_{year}_Q{quarter}.csv"
    filepath = os.path.join(year_dir, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ CSV已保存: {filepath}")
    
    return df


## 2. 批量处理多个年份和季度的数据

In [3]:
# 批量处理多个年份和季度的数据
def batch_process_quarterly(start_year, end_year):
    for year in range(start_year, end_year + 1):
        print(f"\n===== 处理 {year} 年数据 =====")
        year_dfs = []
        
        for quarter in range(1, 5):  # 1-4季度
            try:
                df = download_sentinel1_metadata_by_quarter(year, quarter)
                if df is not None:
                    year_dfs.append(df)
            except Exception as e:
                print(f"❌ 处理 {year}年第{quarter}季度时发生错误: {e}")
                print(traceback.format_exc())
                continue
        
        # 合并该年的所有季度数据
        if year_dfs:
            try:
                combined_df = pd.concat(year_dfs)
                combined_filepath = os.path.join(BASE_OUTPUT_DIR, f"{year}", f"sentinel1_{year}_all.csv")
                combined_df.to_csv(combined_filepath, index=False)
                print(f"✅ {year}年合并CSV已保存: {combined_filepath}")
            except Exception as e:
                print(f"❌ 合并 {year}年数据时发生错误: {e}")
                print(traceback.format_exc())
        else:
            print(f"⚠️ {year}年没有数据可合并")


## 3. 测试单个季度下载

In [4]:
# 测试单个季度下载
# 可以修改年份和季度进行测试
test_year = 2023
test_quarter = 4  # 第3季度 (7-9月)

# 执行测试
test_df = download_sentinel1_metadata_by_quarter(test_year, test_quarter)

# 显示数据前几行
if test_df is not None:
    print(f"\n数据预览 ({test_year}年第{test_quarter}季度):")
    display(test_df.head())


正在查询 2023年第4季度 (Q4: 10-12月) 的Sentinel-1数据...
查询出错: Connection Error (Timeout): CMR took too long to respond. Set asf constant "asf_search.constants.INTERNAL.CMR_TIMEOUT" to increase. (url='https://cmr.earthdata.nasa.gov/search/granules.umm_json', timeout=30)
等待 5 秒后重试...
正在查询 2023年第4季度 (Q4: 10-12月) 的Sentinel-1数据...
找到 5612 条记录
✅ CSV已保存: E:\NWP\Sentinel1\Metadata\2023\sentinel1_2023_Q4.csv

数据预览 (2023年第4季度):


,Granule Name,Platform,Sensor,Beam Mode,Start Time,End Time,Orbit,Path Number,Processing Date,Processing Level,...,Near Start Lat,Near Start Lon,Far Start Lat,Far Start Lon,Far End Lat,Far End Lon,Near End Lat,Near End Lon,Ascending or Descending,URL
0,S1A_IW_RAW__0SDH_20231231T145555_20231231T1456...,Sentinel-1A,C-SAR,IW,2023-12-31T14:55:55Z,2023-12-31T14:56:27Z,51900,28,2023-12-31T14:55:55Z,RAW,...,74.105500,-124.978100,72.182300,-126.282900,71.681500,-119.271800,73.554000,-117.209900,DESCENDING,https://datapool.asf.alaska.edu/RAW/SA/S1A_IW_...
1,S1A_IW_RAW__0SDH_20231231T145555_20231231T1456...,Sentinel-1A,C-SAR,IW,2023-12-31T14:55:55Z,2023-12-31T14:56:27Z,51900,28,2023-12-31T14:55:55Z,METADATA_RAW,...,74.105500,-124.978100,72.182300,-126.282900,71.681500,-119.271800,73.554000,-117.209900,DESCENDING,https://datapool.asf.alaska.edu/METADATA_RAW/S...
2,S1A_IW_SLC__1SDH_20231231T145557_20231231T1456...,Sentinel-1A,C-SAR,IW,2023-12-31T14:55:57Z,2023-12-31T14:56:24Z,51900,28,2023-12-31T14:55:57Z,SLC,...,73.465538,-117.436195,74.056824,-125.247406,72.465691,-126.330933,71.913254,-119.161674,DESCENDING,https://datapool.asf.alaska.edu/SLC/SA/S1A_IW_...
3,S1A_IW_SLC__1SDH_20231231T145557_20231231T1456...,Sentinel-1A,C-SAR,IW,2023-12-31T14:55:57Z,2023-12-31T14:56:24Z,51900,28,2023-12-31T14:55:57Z,METADATA_SLC,...,73.465538,-117.436195,74.056824,-125.247406,72.465691,-126.330933,71.913254,-119.161674,DESCENDING,https://datapool.asf.alaska.edu/METADATA_SLC/S...
4,S1A_IW_RAW__0SDH_20231231T145530_20231231T1456...,Sentinel-1A,C-SAR,IW,2023-12-31T14:55:30Z,2023-12-31T14:56:02Z,51900,28,2023-12-31T14:55:30Z,RAW,...,75.584700,-123.781400,73.666800,-125.297700,73.127900,-117.716200,74.984800,-115.311600,DESCENDING,https://datapool.asf.alaska.edu/RAW/SA/S1A_IW_...


## 4. 批量处理指定年份范围

In [ ]:
# 设置要处理的年份范围
# 注意：处理整个2014-2024年的数据可能需要较长时间
# 建议先测试较小的年份范围，例如2022-2023

start_year = 2022  # 起始年份
end_year = 2023    # 结束年份

print(f"开始批量处理 {start_year}-{end_year} 年的Sentinel-1元数据...")
batch_process_quarterly(start_year, end_year)
print("\n✅ 指定年份范围的数据处理完成!")


## 5. 处理完整的2014-2024年数据

In [ ]:
# 处理完整的2014-2024年数据
# 警告：这将需要较长时间，并可能受到ASF API限制

# 取消下面的注释来执行完整的批处理
# full_start_year = 2014  # Sentinel-1发射于2014年
# full_end_year = 2024
# 
# print(f"开始批量处理完整的 {full_start_year}-{full_end_year} 年的Sentinel-1元数据...")
# batch_process_quarterly(full_start_year, full_end_year)
# print("\n✅ 所有数据处理完成!")


## 6. 查看已下载的数据统计

In [ ]:
# 查看已下载的数据统计
def show_download_statistics():
    print("已下载的Sentinel-1元数据统计:")
    print("===========================")
    
    if not os.path.exists(BASE_OUTPUT_DIR):
        print("未找到数据目录")
        return
    
    years = [d for d in os.listdir(BASE_OUTPUT_DIR) if os.path.isdir(os.path.join(BASE_OUTPUT_DIR, d))]
    years.sort()
    
    total_records = 0
    year_stats = {}
    
    for year in years:
        year_dir = os.path.join(BASE_OUTPUT_DIR, year)
        csv_files = [f for f in os.listdir(year_dir) if f.endswith('.csv')]
        
        year_total = 0
        quarter_stats = {}
        
        for csv_file in csv_files:
            if 'all' in csv_file:
                continue  # 跳过合并文件，避免重复计数
                
            file_path = os.path.join(year_dir, csv_file)
            try:
                df = pd.read_csv(file_path)
                records = len(df)
                
                # 提取季度信息
                if 'Q' in csv_file:
                    quarter = csv_file.split('_')[-1].split('.')[0]
                    quarter_stats[quarter] = records
                    
                year_total += records
            except Exception as e:
                print(f"读取文件 {file_path} 时出错: {e}")
                continue
        
        year_stats[year] = {
            'total': year_total,
            'quarters': quarter_stats
        }
        total_records += year_total
    
    # 打印统计信息
    for year in sorted(year_stats.keys()):
        print(f"\n{year}年: 共 {year_stats[year]['total']} 条记录")
        for quarter, count in sorted(year_stats[year]['quarters'].items()):
            print(f"  - {quarter}: {count} 条记录")
    
    print(f"\n总计: {total_records} 条记录")

# 执行统计
show_download_statistics()
